# 📓 Cuaderno Paso a Paso: Videoportero IP en la Nube (KD9203 + Asterisk VPS + Push + App Móvil)

Este cuaderno interactivo registra **cosa por cosa** cada componente del proyecto y los comandos exactos de instalación y verificación:

1. **Etapa 1**: Cableado físico de 12V, cerradura y red del frente **KD9203**.
2. **Etapa 2**: Preparación del VPS (Ubuntu 22.04 / Debian 12 - 1 vCPU, 2 GB RAM).
3. **Etapa 3**: Instalación de **Asterisk PBX (PJSIP)** con códec de video **H.264**, soporte **NAT** y **WebRTC**.
4. **Etapa 4**: Instalación del **Proxy Push (Flexisip + FCM HTTP v1)** en el mismo VPS para despertar celulares bloqueados.
5. **Etapa 5**: Instalación del **Panel Web Ligero y Generador de Códigos QR** (`:8088/admin`).
6. **Etapa 6**: Configuración web del frente de calle **KD9203**.
7. **Etapa 7**: Configuración y compilación de la **App Móvil (Flutter / Linphone)** y apertura con `#` (DTMF).

## Etapa 1 y 2: Clonar el Repositorio en el VPS
Conéctate por SSH a tu VPS (`ssh root@TU_IP_PUBLICA`) y ejecuta:

In [ ]:
# En el VPS Ubuntu 22.04 / Debian 12:
apt update && apt install -y git
git clone https://github.com/navaliente/portero-ip-vps.git
cd portero-ip-vps
chmod +x scripts/*.sh

## Etapa 3: Instalar Asterisk PBX + Video H.264 + Firewall UFW + Fail2ban
- Configura la extensión `8000` para el frente **KD9203**.
- Configura los departamentos (`101`, `102`, `201`) en doble perfil simultáneo (`SIP estándar` + `WebRTC App`).
- Abre los puertos `5060` (SIP), `5061` (TLS), `8080/8089` (WS/WSS) y `10000-20000` (RTP H.264).

In [ ]:
sudo bash scripts/install_vps.sh
# Ver las contraseñas autogeneradas:
cat /root/credenciales_portero.txt

## Etapa 4: Instalar el Proxy Push en el mismo VPS (Flexisip + FCM HTTP v1)
- Levanta **Flexisip** en los puertos `5062` (TCP/UDP) y `5063` (TLS) conectado a Asterisk en `127.0.0.1:5060`.
- Instala `/var/lib/asterisk/agi-bin/push_wakeup.py` para despertar celulares vía Firebase Cloud Messaging (FCM HTTP v1) antes de timbrar.

In [ ]:
sudo bash scripts/install_flexisip_push.sh

## Etapa 5: Instalar el Panel Web Ligero y Servidor de Códigos QR (Puerto 8088)
- Instala el servicio `portero-panel.service` (`~12 MB RAM`).
- Permite administrar departamentos en `http://TU_IP_VPS:8088/admin` y escanear los códigos QR desde la App del edificio o desde Linphone.

In [ ]:
sudo bash scripts/install_panel_admin.sh
# También puedes agregar departamentos por terminal si lo prefieres:
sudo bash scripts/agregar_vecino.sh 304
sudo bash scripts/generar_qr_vecino.sh 304

## Etapa 6: Parámetros del Frente de Calle KD9203
- **Servidor SIP**: `IP_PUBLICA_DEL_VPS` | **Puerto**: `5060` (`UDP` o `TCP`)
- **Usuario / Auth ID**: `8000` | **Contraseña**: *(Ver `/root/credenciales_portero.txt`)*
- **Códec de Video**: `H.264` (`Baseline Profile`, `720p` o `640x480`, `15 fps`, `512-768 kbps`)
- **Apertura de Puerta (DTMF)**: `RFC2833` $\rightarrow$ Código de relé: `#`

## Etapa 7: Compilar la App Móvil del Edificio (Flutter)
En tu PC de desarrollo, dentro de la carpeta `app-movil`:

In [ ]:
cd app-movil
flutter create . --project-name portero_ip_app
flutter pub get
flutter build apk --release

## Etapa 8: Comandos de Diagnóstico en Vivo en el VPS

In [ ]:
# Ver estado de registro del frente 8000 y de los departamentos:
sudo asterisk -rx "pjsip show aors"

# Ver estado del servicio del Panel Web:
sudo systemctl status portero-panel

# Ver IPs bloqueadas por Fail2ban:
sudo fail2ban-client status asterisk